# Minimal Local Unstructured MCP Server Test Notebook

This notebook installs dependencies, lets you set env vars, starts the HTTP server in a background thread, and tests both `parse_file` and `partition`.

In [ ]:
!apt-get update -y
!apt-get install -y poppler-utils tesseract-ocr libmagic-dev
!pip install -q -r /content/mcp_unstructured/requirements.txt


In [ ]:
from google.colab import files
uploaded = files.upload()
list(uploaded.keys())


In [ ]:
import os, sys
PROJECT_DIR = '/content/mcp_unstructured'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
os.environ['ALLOWED_ROOT'] = '/content'
os.environ['SCANNED_PDF_POLICY'] = 'hi_res'
USER_HI_RES_MODEL_NAME = ''
USER_OCR_AGENT = ''
if USER_HI_RES_MODEL_NAME.strip():
    os.environ['UNSTRUCTURED_HI_RES_MODEL_NAME'] = USER_HI_RES_MODEL_NAME.strip()
else:
    os.environ.pop('UNSTRUCTURED_HI_RES_MODEL_NAME', None)
if USER_OCR_AGENT.strip():
    os.environ['OCR_AGENT'] = USER_OCR_AGENT.strip()
else:
    os.environ.pop('OCR_AGENT', None)
print('ALLOWED_ROOT =', os.environ['ALLOWED_ROOT'])
print('SCANNED_PDF_POLICY =', os.environ['SCANNED_PDF_POLICY'])
print('UNSTRUCTURED_HI_RES_MODEL_NAME =', os.environ.get('UNSTRUCTURED_HI_RES_MODEL_NAME', '<default>'))
print('OCR_AGENT =', os.environ.get('OCR_AGENT', '<default>'))


In [ ]:
import threading
import server
def start_server():
    server.run_http(port=8000)
thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print('HTTP MCP server running on http://localhost:8000')


In [ ]:
uploaded_files = [f'/content/{name}' for name in uploaded.keys()]
sample_path = uploaded_files[0]
print('Sample path:', sample_path)


In [ ]:
import requests
payload = {
    'tool': 'parse_file',
    'path': sample_path,
    'route': 'auto',
    'chunking_strategy': 'by_title',
    'return_elements': True,
    'include_coordinates': False,
    'cleaning_config': {
        'use_clean': True,
        'extra_whitespace': True,
        'group_broken_paragraphs': True,
        'replace_unicode_quotes': True,
        'clean_non_ascii_chars': False,
        'remove_punctuation': False,
    }
}
resp = requests.post('http://localhost:8000', json=payload, timeout=180)
resp.raise_for_status()
result = resp.json()['result']
print('Route used:', result['metadata']['route_used'])
print('Chunking:', result['metadata']['chunking_strategy'])
print('Chunk count:', result['metadata']['num_chunks'])
print('Returned elements:', len(result.get('elements', [])))
print('First chunk page_number:', result['chunks'][0]['page_number'] if result['chunks'] else None)
print('First chunk preview:', result['chunks'][0]['text'][:500] if result['chunks'] else '<no chunks>')


In [ ]:
partition_payload = {
    'tool': 'partition',
    'path': sample_path,
    'route': 'auto',
    'include_coordinates': False,
}
resp2 = requests.post('http://localhost:8000', json=partition_payload, timeout=180)
resp2.raise_for_status()
part = resp2.json()['result']
print('Element count:', part['metadata']['num_elements'])
print('First element:', part['elements'][0] if part['elements'] else '<none>')
